# Atribución sobre el modelo

**Fase:** E9

## Objetivo

Obtener, para cada caso, qué señales y qué tramo temporal sostienen la predicción, con su dirección de contribución.

## Relación con la pregunta de investigación

> ¿Puede un agente con RAG explicar el diagnóstico de un vehículo a partir de datos OBD-II, rastreando cada explicación hasta su evidencia de origen, y reconocer cuándo la evidencia disponible no es suficiente para sostener una explicación en lugar de generar una hipótesis sin fundamento?

Convierte la predicción en evidencia observacional localizable. La atribución no es causalidad y no se presentará como tal.

## Hipótesis

_Por declarar antes de ejecutar la fase._

## Datos

_Dataset, versión y partición utilizados._

## Método

_Procedimiento, parámetros y semillas._

## Métricas

_Qué se mide y con qué instrumento._

## Criterio de éxito o refutación

_Fijado antes de mirar los resultados._


In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd

from mecanico_ia_exp.config import paths


## Atribución sobre el modelo especializado

La función explicada se declara siempre. El fondo sale solo de entrenamiento.

In [2]:
import json
x = json.loads((paths.EXPERIMENTS / 'E9_xai_synthetic_v2.json').read_text())
print('función explicada :', x['explained_function'])
print('explainer         :', x['explainer'])
print('fondo             :', x['background'])

función explicada : predict_proba[clase_predicha]
explainer         : shap.explainers.Permutation
fondo             : {'n': 100, 'source': 'train, clase normal', 'hash': '8757d917835e3981', 'seed': 20260901}


### El hallazgo: `maf` no contribuye a detectar su propio fallo

In [3]:
atr = pd.read_csv(paths.TABLES / 'E9_atribuciones_synthetic_v2.csv')
f02 = atr[atr.predicted_class == 'F02_maf_drift']
(f02.groupby('signal')['importance']
    .apply(lambda s: s.abs().sum()).sort_values(ascending=False)
    .round(4).to_frame('importancia acumulada'))

,importancia acumulada
signal,
ltft_bank1,0.8753
stft_bank1,0.6185
stft_bank1+ltft_bank1,0.1583
map,0.0766


In [4]:
from mecanico_ia_exp.obd.faults import FAULTS
print('la ficha del fallo declara:', FAULTS['F02_maf_drift'].affected_signals)
print()
print('La atribución NO señala maf. Es el hallazgo de E4 por otra vía.')
print('Una explicación que dijera «el caudal es bajo» sería falsa.')

la ficha del fallo declara: ('maf', 'stft_bank1', 'ltft_bank1')

La atribución NO señala maf. Es el hallazgo de E4 por otra vía.
Una explicación que dijera «el caudal es bajo» sería falsa.


> **La atribución no es causalidad.** Dice de qué depende la salida del modelo, no qué le ocurre al vehículo.

## Resultados

_Lo observado, sin interpretación._

## Interpretación

_Qué significa, y qué no significa._

## Limitaciones

_Qué queda fuera del alcance de lo mostrado._

## Conclusión experimental

_Respuesta al criterio fijado en la cabecera._

## Artefactos generados

_Ficheros producidos, con su ruta._
